# Energy Consumption Dataset — Data Cleaning

**Internship task:** Identify and handle missing values, duplicate records, incorrect data types, and inconsistent values.

This notebook uses Pandas. The original dataset is preserved in `dataset/raw_dataset.csv`; the cleaned output is saved as `dataset/cleaned_dataset.csv`.

> In Google Colab, upload `raw_dataset.csv` when prompted, or place it in a folder named `dataset` alongside this notebook.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Try the project folder first. If it is not found, upload the CSV in Colab.
raw_path = Path("dataset/raw_dataset.csv")
if not raw_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        raw_path = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError("Place raw_dataset.csv inside the dataset folder.")

df = pd.read_csv(raw_path)
print("Dataset shape:", df.shape)
display(df.head())

## 1. Initial inspection

In [ ]:
df.info()
display(df.describe(include="all").T)
print("\\nMissing values by column:")
display(df.isna().sum())
print("Duplicate full rows:", df.duplicated().sum())

## 2. Check categorical consistency and timestamp quality

In [ ]:
categorical_cols = ["HVACUsage", "LightingUsage", "DayOfWeek", "Holiday"]
for col in categorical_cols:
    print(f"\\n{col} values:")
    print(df[col].value_counts(dropna=False))
    print("Rows with leading/trailing spaces:",
          df[col].astype("string").ne(df[col].astype("string").str.strip()).sum())

parsed_ts = pd.to_datetime(df["Timestamp"], format="%d-%m-%Y %H:%M", errors="coerce")
print("\\nTimestamp parse failures:", parsed_ts.isna().sum())
print("Duplicate timestamps:", parsed_ts.duplicated().sum())
weekday_mismatch = parsed_ts.dt.day_name().ne(
    df["DayOfWeek"].astype("string").str.strip()
)
print("Weekday mismatches:", weekday_mismatch.sum())

## 3. Clean the dataset

- Convert `Timestamp` from text to datetime.
- Trim and standardize categorical text.
- Recalculate `DayOfWeek` from `Timestamp`, because the timestamp is the more reliable date field and 879 weekday labels did not match it.
- Ensure measurement columns are numeric.
- No missing-value imputation or duplicate deletion is needed because the initial inspection found none.

In [ ]:
clean = df.copy()

clean["Timestamp"] = pd.to_datetime(
    clean["Timestamp"], format="%d-%m-%Y %H:%M", errors="coerce"
)

for col in ["HVACUsage", "LightingUsage", "DayOfWeek", "Holiday"]:
    clean[col] = clean[col].astype("string").str.strip().str.title()

# Rebuild weekday labels from the parsed timestamp.
clean["DayOfWeek"] = clean["Timestamp"].dt.day_name()

numeric_cols = [
    "Temperature", "Humidity", "SquareFootage", "Occupancy",
    "RenewableEnergy", "EnergyConsumption"
]
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

display(clean.head())
clean.info()

## 4. Validate the cleaned dataset

In [ ]:
print("Missing values after cleaning:")
display(clean.isna().sum())
print("Duplicate rows after cleaning:", clean.duplicated().sum())
print("Timestamp dtype:", clean["Timestamp"].dtype)
print("Weekday mismatches after cleaning:",
      clean["Timestamp"].dt.day_name().ne(clean["DayOfWeek"]).sum())
print("Negative numeric values:")
display((clean[numeric_cols] < 0).sum())

## 5. Export cleaned dataset

In [ ]:
output_dir = Path("dataset")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "cleaned_dataset.csv"
clean.to_csv(output_path, index=False, date_format="%d-%m-%Y %H:%M")
print("Saved:", output_path.resolve())

# In Google Colab, download the cleaned CSV if desired.
try:
    from google.colab import files
    files.download(str(output_path))
except ImportError:
    pass